## Data Quality Analysis (Missing, Duplicates and Data Types fixes)

In [231]:
import pandas as pd
import numpy as np
from configs.settings import project_dir

In [232]:
p_dir = project_dir()

#path to raw data files
raw_data_dir = p_dir.RAW_DATA_DIR
customers = raw_data_dir/'olist_customers_dataset.csv'
location = raw_data_dir/'olist_geolocation_dataset.csv'
items = raw_data_dir/'olist_order_items_dataset.csv'
payments = raw_data_dir/'olist_order_payments_dataset.csv'
reviews = raw_data_dir/'olist_order_reviews_dataset.csv'
orders = raw_data_dir/'olist_orders_dataset.csv'
products = raw_data_dir/'olist_products_dataset.csv'
sellers = raw_data_dir/'olist_sellers_dataset.csv'
category = raw_data_dir /'product_category_name_translation.csv'

### Customers Dataset 

In [233]:
customers_df = pd.read_csv(customers)
customers_df.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [234]:
# Missing Values
print(customers_df.isnull().sum())
print(f"\n\nNumber of rows with duplicates rows: {customers_df.duplicated().sum()}\n\n")
customers_df.info()

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64


Number of rows with duplicates rows: 0


<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 3.8 MB


In [235]:
print(f"Number of duplicates value in customer_city col is {customers_df['customer_city'].unique().duplicated().sum()}\n")
print(f"Number of duplicates values in customer state is {customers_df['customer_state'].value_counts().duplicated().sum()}")

customers_df['customer_zip_code_prefix'].value_counts().sort_values(ascending=False).to_string('../arson/zip_code.txt')

Number of duplicates value in customer_city col is 0

Number of duplicates values in customer state is 0


### Geolocation Dataset

In [236]:
location_df = pd.read_csv(location)
location_df.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP


In [237]:
location_df.shape

(1000163, 5)

In [238]:
# print(location_df.isnull().sum())
# print(f"Number of Duplicates in location dataset of {location_df['geolocation_city'].value_counts().sort_values().to_string('../arson/location_city.txt')}")

location_df['geolocation_city'] = location_df['geolocation_city'].astype('string')
location_df['geolocation_city'] = location_df['geolocation_city'].str.strip()
# location_df['geolocation_city'] = location_df['geolocation_city'].str.replace('`','',regex='False')
# location_df['geolocation_city'] = location_df['geolocation_city'].str.replace('^','',regex='False')
# location_df['geolocation_city'] = location_df['geolocation_city'].str.replace('~','',regex='False')
# location_df['geolocation_city'].value_counts().sort_values().to_string('../arson/location_city_clean.txt')

In [239]:
location_df[['geolocation_lat','geolocation_lng']].duplicated().sum()

np.int64(281700)

In [240]:
location_df['geolocation_city'].value_counts().sort_values().to_string('../arson/lat.csv')


In [241]:
location_df.head()
location_df.info()


<class 'pandas.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  int64  
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-null  float64
 3   geolocation_city             1000163 non-null  string 
 4   geolocation_state            1000163 non-null  str    
dtypes: float64(2), int64(1), str(1), string(1)
memory usage: 38.2 MB


In [242]:
messy_data = location_df[location_df[['geolocation_lat','geolocation_lng']].duplicated()]
# messy_data[['geolocation_city','geolocation_lat','geolocation_lng']].value_counts().head(30)
messy_data['geolocation_city'] = messy_data['geolocation_city'].astype(str)
messy_data['geolocation_city'] = messy_data['geolocation_city'].str.strip()
messy_data[['geolocation_lat','geolocation_lng','geolocation_city']].groupby(['geolocation_lat','geolocation_lng']).value_counts().sort_values(ascending=False).to_string('../arson/messy1.csv')

In [243]:
messy_data[['geolocation_lat','geolocation_lng']].duplicated()

15         False
44          True
65          True
66         False
67          True
           ...  
1000153     True
1000154     True
1000159    False
1000160    False
1000162     True
Length: 281700, dtype: bool

In [244]:
messy_data[messy_data['geolocation_city'] == 'xambre']

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
897964,87535,-23.729684,-53.489767,xambre,PR
898303,87535,-23.729684,-53.489767,xambre,PR
898430,87535,-23.733100,-53.488682,xambre,PR
898450,87535,-23.737584,-53.485975,xambre,PR


In [245]:
messy_data[['geolocation_city','geolocation_lat','geolocation_lng']][messy_data[['geolocation_lat','geolocation_lng']].duplicated()].head(30).sort_values(by=['geolocation_lat','geolocation_lng'])

,geolocation_city,geolocation_lat,geolocation_lng
237,sao paulo,-23.552496,-46.632060
337,sao paulo,-23.552235,-46.628441
136,sao paulo,-23.549854,-46.643139
161,sao paulo,-23.549854,-46.643139
223,sao paulo,-23.549854,-46.643139
240,sao paulo,-23.549854,-46.643139
280,sao paulo,-23.549819,-46.635606
253,sao paulo,-23.546935,-46.636588
275,sao paulo,-23.546935,-46.636588
306,são paulo,-23.546935,-46.636588


In [246]:
messy_data[['geolocation_lat','geolocation_lng','geolocation_city']].where(messy_data[['geolocation_lat','geolocation_lng']].duplicated())

,geolocation_lat,geolocation_lng,geolocation_city
15,NaN,NaN,NaN
44,-23.546081,-46.644820,sao paulo
65,-23.546081,-46.644820,sao paulo
66,NaN,NaN,NaN
67,-23.546081,-46.644820,sao paulo
...,...,...,...
1000153,-28.343273,-51.873734,ciriaco
1000154,-28.070493,-52.011342,tapejara
1000159,NaN,NaN,NaN
1000160,NaN,NaN,NaN


In [247]:
messy_data[messy_data['geolocation_zip_code_prefix'].duplicated()]

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
44,1046,-23.546081,-46.644820,sao paulo,SP
65,1046,-23.546081,-46.644820,sao paulo,SP
67,1046,-23.546081,-46.644820,sao paulo,SP
72,1046,-23.545320,-46.644069,sao paulo,SP
82,1046,-23.546081,-46.644820,sao paulo,SP
...,...,...,...,...,...
1000153,99970,-28.343273,-51.873734,ciriaco,RS
1000154,99950,-28.070493,-52.011342,tapejara,RS
1000159,99900,-27.877125,-52.224882,getulio vargas,RS
1000160,99950,-28.071855,-52.014716,tapejara,RS


In [248]:
location_df[location_df['geolocation_zip_code_prefix'].duplicated()].sort_values(by='geolocation_zip_code_prefix').head(10)

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
1384,1001,-23.549292,-46.633559,sao paulo,SP
206,1001,-23.550498,-46.634338,sao paulo,SP
1351,1001,-23.549951,-46.634027,são paulo,SP
235,1001,-23.550642,-46.634410,sao paulo,SP
985,1001,-23.550498,-46.634338,sao paulo,SP
1004,1001,-23.549292,-46.633559,sao paulo,SP
575,1001,-23.549779,-46.633957,são paulo,SP
519,1001,-23.551337,-46.634027,sao paulo,SP
1062,1001,-23.550498,-46.634338,sao paulo,SP
299,1001,-23.549698,-46.633909,sao paulo,SP


In [249]:
location_df_clean = location_df.drop_duplicates(subset=['geolocation_lat','geolocation_lng'])

In [250]:
print(location_df.shape)
print(location_df_clean.shape)

(1000163, 5)
(718463, 5)


In [251]:
1000163 - 718463

281700

In [252]:
location_df.duplicated().sum()

np.int64(261831)

In [253]:
location_df_clean.duplicated().sum()

np.int64(0)

## Order Items dataset cleaning

In [254]:
order_items_df = pd.read_csv(items)
order_items_df.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [255]:
order_items_df.info()
# No null values, only incorrect dtype of date column

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  str    
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  str    
 3   seller_id            112650 non-null  str    
 4   shipping_limit_date  112650 non-null  str    
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), str(4)
memory usage: 6.0 MB


In [256]:
order_items_df['order_item_id'].value_counts()
# order_items_df.duplicated().sum()

order_item_id
1     98666
2      9803
3      2287
4       965
5       460
6       256
7        58
8        36
9        28
10       25
11       17
12       13
13        8
14        7
15        5
16        3
17        3
18        3
19        3
20        3
21        1
Name: count, dtype: int64

In [257]:
# fixing the dtype of date col
order_items_df['shipping_limit_date'] = order_items_df['shipping_limit_date'].astype('datetime64[ns]')

In [258]:
order_items_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   order_id             112650 non-null  str           
 1   order_item_id        112650 non-null  int64         
 2   product_id           112650 non-null  str           
 3   seller_id            112650 non-null  str           
 4   shipping_limit_date  112650 non-null  datetime64[ns]
 5   price                112650 non-null  float64       
 6   freight_value        112650 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(1), str(3)
memory usage: 6.0 MB


### Order Payments Dataset cleaning

In [259]:
order_payments_df = pd.read_csv(payments)
order_payments_df.sample(10)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
29404,85194cc4465ab41c762612c2d288ce8e,1,credit_card,7,220.24
47600,76f9204ee569a062f27c7e6e5c51983c,1,credit_card,2,198.99
83395,f2fd4e0fc194bc29e11d7d3f83151996,1,credit_card,1,35.69
39434,68d1af4ddb903ed2dc66559f1d6ccb6d,1,credit_card,1,191.18
58220,7a301f27a543e77e6176456966c66e2c,1,boleto,1,30.79
45165,f1f4bec57caa5671e6bbd650b5872beb,1,credit_card,6,64.00
17364,6ddda171d214be5f13055479d8b41364,1,credit_card,6,364.99
102119,6be7d999fa8bac782218f8e6281ff562,1,credit_card,3,119.63
45293,f8204e84e9a426029a3f9848738caf33,1,credit_card,5,118.48
60465,13e7a676a3096c762e0f6f7d63a48e6b,1,credit_card,3,181.16


In [260]:
# order_payments_df.info()
# order_payments_df['payment_type'].value_counts()
order_payments_df.groupby('payment_type')['payment_installments'].value_counts()

payment_type  payment_installments
boleto        1                       19784
credit_card   1                       25455
              2                       12413
              3                       10461
              4                        7098
              10                       5328
              5                        5239
              8                        4268
              6                        3920
              7                        1626
              9                         644
              12                        133
              15                         74
              18                         27
              11                         23
              24                         18
              20                         17
              13                         16
              14                         15
              17                          8
              16                          5
              21                         

In [261]:
order_payments_df.duplicated().sum()

np.int64(0)

In [262]:
## replace boleto with wallet, handled not_defined values with median
order_payments_df['payment_type'] = order_payments_df['payment_type'].str.replace('boleto','wallet')

order_payments_df['payment_type'] = order_payments_df['payment_type'].str.replace('not_defined',order_payments_df['payment_type'].mode()[0])

In [263]:
order_payments_df['payment_type'].value_counts()

payment_type
credit_card    76798
wallet         19784
voucher         5775
debit_card      1529
Name: count, dtype: int64

## Order Reviews Dataset cleaning

In [264]:
order_reviews_df = pd.read_csv(reviews)
order_reviews_df.sample(10)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
13199,d96d6316e6830be91bffda6c7b103de0,911cd0cf037580ad6a00dfd5690bda12,5,NaN,Só acho que se fosse entregue através de trans...,2017-04-13 00:00:00,2017-04-16 02:02:19
26089,77700ed6c3ac2e82a4fa50d5c20d4bcf,b4d84c94b807c15d5499ce40b6122efd,4,NaN,NaN,2017-05-17 00:00:00,2017-05-20 11:00:08
75422,3a84e9c87596aa6c932e061ac74b424f,4d9c8dea9380270c4d8151980f6c579c,5,NaN,NaN,2018-02-02 00:00:00,2018-02-05 05:41:44
87073,b8a5fabf78dad94a57610e6835fc9dd1,99ad48402644a3968f2481defdc57947,1,NaN,Um dos produtos marcado como entregue não foi ...,2017-08-11 00:00:00,2017-08-15 01:26:05
6670,9412adb8f11d402d3ae696dc8f496467,fe51b7fac3bea42115fa1c3b8973cba7,5,NaN,NaN,2017-12-19 00:00:00,2017-12-20 14:00:20
27661,26012cfd25d6d032f43b7cfe268367fe,ed6c99480bc1ab87159e288140055645,5,Recomendo,Gostei da balança.,2018-05-12 00:00:00,2018-05-13 11:39:34
8998,bb0b4445ab35264189ce2c33cf8c83aa,54c4b0926f9b4aeb54164306d1374e8f,5,NaN,NaN,2018-01-17 00:00:00,2018-01-19 21:17:28
95669,d619442dbe62c8a4708722ebc1f96036,6f199c0873866afaadffaa53f4808a21,1,NaN,"RECOMENDO A LOJA, MAS O PERFUME NÃO. TEM CHEIR...",2017-08-22 00:00:00,2017-08-23 12:38:59
2998,74b2b4f91de0e6e32992cb4c90e84e5a,b94cb06da9bf9f6f9b4324b294666f4d,5,NaN,NaN,2018-02-08 00:00:00,2018-02-09 10:19:32
58330,4ac382f18cfa3b46cb9ef804b167078a,3343de372beb853bffdaa61da97a22aa,4,NaN,NaN,2018-06-27 00:00:00,2018-06-29 14:53:54


In [265]:
order_reviews_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   review_id                99224 non-null  str  
 1   order_id                 99224 non-null  str  
 2   review_score             99224 non-null  int64
 3   review_comment_title     11568 non-null  str  
 4   review_comment_message   40977 non-null  str  
 5   review_creation_date     99224 non-null  str  
 6   review_answer_timestamp  99224 non-null  str  
dtypes: int64(1), str(6)
memory usage: 5.3 MB


In [ ]:
order_reviews_df.duplicated().sum()
order_reviews_df['review_score'].value_counts()
# TO-DO
## handle the date dtype col, handle missing value in both title and comment message.(a better approach to handle, cant drop all these rows)

review_score
5    57328
4    19142
1    11424
3     8179
2     3151
Name: count, dtype: int64

In [267]:
# fixing the date time dtype col
order_reviews_df['review_creation_date'] = order_reviews_df['review_creation_date'].astype('datetime64[ns]')
order_reviews_df['review_answer_timestamp'] = order_reviews_df['review_answer_timestamp'].astype('datetime64[ns]')

In [268]:
print(f"duplicate review id in table is {order_reviews_df['review_id'].duplicated().sum()}")
order_reviews_df = order_reviews_df.drop_duplicates(subset='review_id',keep='last')
# order_reviews_df.groupby('review_id')[['review_comment_title','review_comment_message']].value_counts().to_string('../arson/review_id.csv')
order_reviews_df.shape

duplicate review id in table is 814


(98410, 7)

In [ ]:
print(order_reviews_df.isnull().sum())
## TO-DO: need to analysis these missing values and an approach to handle these values



review_id                      0
order_id                       0
review_score                   0
review_comment_title       86891
review_comment_message     57742
review_creation_date           0
review_answer_timestamp        0
dtype: int64


In [278]:
order_reviews_df[['review_id','review_comment_title','review_comment_message']].sample(20)
# checking whether

,review_id,review_comment_title,review_comment_message
61478,64d645df62274ad1d62643c6663a0fb3,NaN,O produto não condiz com a descrição. É de bai...
89738,92f33c4c6c18a4374da1f766ebd3a16c,NaN,"como sempre, ótima compra!produto de qualidade..."
81751,312327a3f1d8387a2c36bcb14d65c272,NaN,Comprei o controle para ar condicionado da kom...
62767,85e2af4a10986f6fe1cc91f4362ba4d7,NaN,NaN
44304,4c5b34241638af9e9fe3e4257bc38cbc,NaN,NaN
3813,2c5f0d3a7d7f0894506d523288972cc3,NaN,NaN
49920,aa386f1ffbd14c26b17ce4dca1396856,NaN,NaN
6783,c33f3c03686fdba8c6cf919dd56aac6f,NaN,"Super macia, adorei recomendo sim"
24911,f62f561d340edddfc90d823b2277fda4,NaN,NaN
79841,3fe9b6c3d5e79ebc066c6d466cf0d504,NaN,NaN


In [294]:

order_reviews_df[order_reviews_df['review_comment_message'].isna()][['review_id','review_comment_title','review_comment_message','review_score']]

,review_id,review_comment_title,review_comment_message,review_score
0,7bc2406110b926393aa56f80a40eba40,NaN,NaN,4
1,80e641a11e56f04c1ad469d5645fdfde,NaN,NaN,5
2,228ce5500dc1d8e020d8d1322874b6f0,NaN,NaN,5
5,15197aa66ff4d0650b5434f1b46cda19,NaN,NaN,1
6,07f9bee5d1b850860defd761afa7ff16,NaN,NaN,5
...,...,...,...,...
99217,c6b270c61f67c9f7cb07d84ea8aeaf8b,NaN,NaN,5
99218,af2dc0519de6e0720ef0c74292fb4114,NaN,NaN,5
99219,574ed12dd733e5fa530cfd4bbf39d7c9,NaN,NaN,5
99220,f3897127253a9592a73be9bdfdf4ed7a,NaN,NaN,5


In [308]:
## filling title and message with no comment
order_reviews_df[['review_comment_title','review_comment_message']] = order_reviews_df[['review_comment_title','review_comment_message']].fillna('No Comment')

In [317]:
# creating a new col has_commnet or not
order_reviews_df['has_comment'] = (order_reviews_df['review_comment_title'] != 'No Comments')

In [318]:
order_reviews_df.sample(10)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,has_comment
29237,96165c3065ec553cddd45848b6f64ab0,39b18d551e1accb764651ec6d8abd39f,1,No Comments,Só recebi o produto pq fui buscar nós Correios...,2018-04-24,2018-04-24 20:09:34,False
45636,27c59403ffc24f40df5b5c6a4379bf2f,6574317cd3327f737e39fd9a77969ef2,5,Entrega de produto,Shptime sempre 100%.,2018-07-10,2018-07-11 11:57:24,True
28207,428dabb6cfd7c308a3808cc7d70c6918,a8f2b9f64cddf499563efdb804537781,1,No Comments,Adquiri dois cartuchos HP e a impressora detec...,2018-03-21,2018-03-22 12:15:21,False
37726,84da81b31ceebea3c9058ac29854833e,befe8fab6c34f5861b73a3645ce38e24,5,No Comments,No Comments,2018-05-10,2018-05-12 07:50:51,False
32915,c6a2b17849ccd547eb43ab601acda1ab,ac3143c0b6db91dd3ca4ab4d8a8243eb,5,No Comments,No Comments,2018-08-28,2018-08-30 20:12:19,False
15876,a41c53280f65a2851f2940fda6878e07,203b4c4ffae20ac2c0e84994938927aa,5,No Comments,No Comments,2018-04-13,2018-04-14 01:09:58,False
26388,245f422e5e9ed1dc4123e721d49b0d66,d37222a038dff3570139ecf1b452dd3d,5,No Comments,Muito bom gostei de mais no começo eu fiquei c...,2017-10-11,2017-10-16 18:53:03,False
66705,76981e9d68c67dbf5bb374bcfdca3e1a,5d93a32a25ed8f8e884e5a1d283c43dd,1,No Comments,No Comments,2018-07-11,2018-07-11 18:55:16,False
20926,32c7a9042e7faa961295d4f174d10ff8,f97ad7d35270638277b3209506431b82,5,Produto de alta qualidade,"Amei o material! Protege bem , com alças ajust...",2018-05-17,2018-05-21 11:13:45,True
83304,12d3ce4e7a5f507828a620c4de639830,dfe1a1cff6291723d37ed8f60e12d5c9,5,No Comments,No Comments,2018-08-07,2018-08-10 02:23:35,False


In [319]:
order_reviews_df.isnull().sum()

review_id                  0
order_id                   0
review_score               0
review_comment_title       0
review_comment_message     0
review_creation_date       0
review_answer_timestamp    0
has_comment                0
dtype: int64

In [320]:
order_reviews_df[['review_comment_title','review_comment_message']].value_counts().to_string('../arson/comments.csv')